#### Simple Tokenization (V1 and V2) of the verdict short story

In [4]:
file_path = "the-verdict.txt" # phase-1-foundations/tokenization-chap2/the-verdict.txt
with open(file_path, "r") as file:
    raw_text = file.read()
print(f"Total number of characters: {len(raw_text)}")
print(raw_text[:99])  # first 100 characters 

Total number of characters: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


In [9]:
import re

text = "Hello world. This, and this, is a test."
result = re.split(r'(\s)', text)
# remove whitespace for now, include in later tokeization techniques
result = [item for item in result if item.strip() != ''] 
print(result)

['Hello', 'world.', 'This,', 'and', 'this,', 'is', 'a', 'test.']


In [ ]:
import re

# handle more punctuation.
text = "Hello, world. Is this-- a test?"
result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item.strip() for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


In [12]:
# Now, let us apply it to the short story
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(len(preprocessed)) # first 30 tokenized words.

4690


In [ ]:
print(preprocessed[:30]) # first 30 tokenized words.

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


In [17]:
# Let us find the unique words, which is vocabulary
vocab = sorted(set(preprocessed))
vocab_size = len(vocab)
print(f"Vocabulary size: {vocab_size}")

Vocabulary size: 1130


In [23]:
vocab = {token:integer for integer,token in enumerate(vocab)}
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 50:
        break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)


In [34]:
# Create an inverse version of the tokenizer, a dictionary that maps integers to tokens 
# based on the vocab dict above.
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()} 

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
    
    def decode(self, ids):        
            text = " ".join([self.int_to_str[i] for i in ids]) 
            
            text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)   
            return text

In [35]:
# Test the simple tknizer class
tokenizer = SimpleTokenizerV1(vocab)
test_text =  """"It's the last he painted, you know," 
       Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(test_text)
print(f"Encoded ids: {ids}")

Encoded ids: [1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


In [27]:
# Reverse:
print(f"Decoded text from ids: {tokenizer.decode(ids)}")

Decoded text from ids: " It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


In [29]:
# Add special context tokens:
vocab = sorted(set(preprocessed))
vocab.extend(["<|endoftext|>", "<|unk|>"])
vocab = {token:integer for integer,token in enumerate(vocab)}
print(len(vocab.items()))

1132


In [36]:
# Updating the tknizer to include spl and unknown tkns.
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items()}
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        preprocessed = [item if item in self.str_to_int           
                        else "<|unk|>" for item in preprocessed]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)   
        return text

In [39]:
# test
text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."
text = f"{text1} <|endoftext|> {text2}"

tokenizer2 = SimpleTokenizerV2(vocab)
ids = tokenizer2.encode(text)
print(f"Encoded ids: {ids}")

Encoded ids: [1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]


In [40]:
# reverse:
print(tokenizer2.decode(ids))

<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.


#### Byte Pair Encoding Tokenizer

In [ ]:
# ! uv pip install tiktoken

Using Python 3.12.12 environment at: /Users/sesha/Personal_projects/llm-engg/.venv
Resolved 7 packages in 425ms                                         
Prepared 4 packages in 106ms                                                 tiktoken             ------------------------------ 908.16 KiB/1014.16 KiB      tiktoken             ------------------------------ 16.00 KiB/1014.16 KiB       tiktoken             ------------------------------ 16.00 KiB/1014.16 KiB       
Installed 4 packages in 3ms3.5.1                            
 + charset-normalizer==3.5.1
 + requests==2.34.2
 + tiktoken==0.14.0
 + urllib3==2.7.0


In [7]:
from importlib.metadata import version

import tiktoken

print(f"tiktoken version: {version('tiktoken')}")


tiktoken version: 0.14.0


In [8]:
# instantiate
tokenizer = tiktoken.get_encoding("gpt2")

In [17]:
# test
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     " of someunknownPlace."
)
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 286, 617, 34680, 27271, 13]


In [18]:
strings = tokenizer.decode(integers)
print(strings)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace.


In [20]:
# Exercise 2.1
text_exercise = "Akwirw ier"
integers_exercise = tokenizer.encode(text_exercise, allowed_special={"<|endoftext|>"})
strings_exercise = tokenizer.decode(integers_exercise)
print(f"encoded: {integers_exercise}\ndecoded: {strings_exercise}")

encoded: [33901, 86, 343, 86, 220, 959]
decoded: Akwirw ier
